# SGD, SCG and LeapFrog in PyTorch

The same experiment as `1_main_experiment.ipynb`, with the network and the three training
algorithms written in PyTorch. The tensors use 64-bit floats, the starting weights come from the
same NumPy random numbers and the gradient uses the same backpropagation formulas, so the results
should match the NumPy version. Autograd checks the gradient, and the last section prints the
largest differences from the NumPy results.

### Contents
1. Settings
2. The network
3. The training algorithms
4. The problems and the folds
5. Step 1: the number of hidden units
6. Step 2: the control parameters
7. Step 3: five-fold evaluation
8. Friedman and Nemenyi tests
9. Convergence
10. Comparison with the NumPy version

Steps 1 and 2 take about an hour. With `RUN_SEARCH = False` the notebook reads the search results
of `1_main_experiment.ipynb` from `results/` and only repeats Step 3 in PyTorch.
New results are saved in `results/pytorch/`.

## 1. Settings

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy import stats
from sklearn import datasets as ds
from sklearn.model_selection import KFold, StratifiedKFold

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)

ALGORITHMS = ["SGD", "SCG", "LeapFrog"]
PROBLEMS = ["wine", "breast_cancer", "digits", "diabetes", "airfoil", "california"]
HIDDEN_UNIT_CANDIDATES = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128]
N_FOLDS = 5
MAX_EPOCHS = 200        # gradient calculations per training run
MAX_PATTERNS = 5000     # size cap for Steps 1 and 2
SEED = 0                # folds of Step 1
TUNE_SEED = 1           # folds of Step 2
FINAL_SPLIT_SEED = 42   # folds of Step 3

# Steps 1 and 2 take about an hour. With RUN_SEARCH = False the notebook reads the search
# results of 1_main_experiment.ipynb from results/ and only repeats Step 3 in PyTorch.
RUN_SEARCH = False
MAIN_RESULTS = "results"                 # results of 1_main_experiment.ipynb
RESULTS = os.path.join("results", "pytorch")
os.makedirs(RESULTS, exist_ok=True)
COLOURS = {"SGD": "#1769aa", "SCG": "#c36c18", "LeapFrog": "#008675"}

## 2. The network

In [ ]:
def make_network(n_inputs, n_hidden, n_outputs, task):
    """Store the layer sizes and the task (classification or regression)."""
    return {"n_inputs": n_inputs, "n_hidden": n_hidden, "n_outputs": n_outputs, "task": task}


def init_weights(net, seed):
    """The same starting weights as the NumPy version, as a tensor."""
    rng = np.random.RandomState(seed)
    n_in, n_hid, n_out = net["n_inputs"], net["n_hidden"], net["n_outputs"]
    W1 = rng.uniform(-1, 1, size=(n_hid, n_in)) / np.sqrt(n_in)
    b1 = rng.uniform(-1, 1, size=n_hid) / np.sqrt(n_in)
    W2 = rng.uniform(-1, 1, size=(n_out, n_hid)) / np.sqrt(n_hid)
    b2 = rng.uniform(-1, 1, size=n_out) / np.sqrt(n_hid)
    return torch.tensor(np.concatenate([W1.ravel(), b1, W2.ravel(), b2]))


def unpack_weights(w, net):
    """Split the flat weight vector into W1, b1 (hidden layer) and W2, b2 (output layer)."""
    n_in, n_hid, n_out = net["n_inputs"], net["n_hidden"], net["n_outputs"]
    a = n_hid * n_in
    b = a + n_hid
    c = b + n_out * n_hid
    return w[:a].reshape(n_hid, n_in), w[a:b], w[b:c].reshape(n_out, n_hid), w[c:]


def forward(w, X, net):
    """Return the hidden activations H and the network outputs Y."""
    W1, b1, W2, b2 = unpack_weights(w, net)
    H = torch.tanh(X @ W1.T + b1)
    Z = H @ W2.T + b2
    if net["task"] == "classification":
        Z = Z - Z.max(dim=1, keepdim=True).values
        expZ = torch.exp(Z)
        return H, expZ / expZ.sum(dim=1, keepdim=True)
    return H, Z


def compute_loss(w, X, T, net):
    """Cross-entropy for classification and half the mean squared error for regression."""
    H, Y = forward(w, X, net)
    n = X.shape[0]
    if net["task"] == "classification":
        return -torch.sum(T * torch.log(Y + 1e-12)) / n
    return 0.5 * torch.sum((Y - T) ** 2) / n


def compute_gradient(w, X, T, net):
    """Backpropagation: the gradient of compute_loss with respect to the weight vector."""
    W1, b1, W2, b2 = unpack_weights(w, net)
    H, Y = forward(w, X, net)
    delta_out = (Y - T) / X.shape[0]
    delta_hidden = delta_out @ W2 * (1 - H ** 2)
    return torch.cat([(delta_hidden.T @ X).reshape(-1), delta_hidden.sum(dim=0),
                      (delta_out.T @ H).reshape(-1), delta_out.sum(dim=0)])


def compute_error(w, X, T, net):
    """Misclassification rate for classification and mean squared error for regression."""
    with torch.no_grad():
        H, Y = forward(w, X, net)
        if net["task"] == "classification":
            return (Y.argmax(dim=1) != T.argmax(dim=1)).double().mean().item()
        return ((Y - T) ** 2).mean().item()


def test_loss(w, X, T, net):
    """Cross-entropy for classification and the full MSE for regression."""
    with torch.no_grad():
        if net["task"] == "classification":
            return compute_loss(w, X, T, net).item()
        return compute_error(w, X, T, net)

Check the backpropagation gradient against autograd.

In [ ]:
torch.manual_seed(1)
for task in ["classification", "regression"]:
    X = torch.randn(20, 4)
    T = torch.eye(3)[torch.randint(0, 3, (20,))] if task == "classification" else torch.randn(20, 1)
    net = make_network(4, 5, T.shape[1], task)
    w = init_weights(net, 0).requires_grad_(True)
    autograd = torch.autograd.grad(compute_loss(w, X, T, net), w)[0]
    print(task, "largest difference:", (compute_gradient(w.detach(), X, T, net) - autograd).abs().max().item())

## 3. The training algorithms
The same algorithms as in `1_main_experiment.ipynb`: 200 gradient calculations per run.

In [ ]:
def train_sgd(w, X, T, net, learning_rate, momentum, batch_size, max_epochs, seed, callback=None):
    """Mini-batch gradient descent with momentum. One epoch is one pass over the training set."""
    rng = np.random.RandomState(seed)
    w = w.clone()
    velocity = torch.zeros(len(w))
    n = X.shape[0]
    for epoch in range(max_epochs):
        order = torch.tensor(rng.permutation(n))
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            gradient = compute_gradient(w, X[batch], T[batch], net)
            velocity = momentum * velocity - learning_rate * gradient
            w = w + velocity
        if callback:
            callback(w)
    return w

In [ ]:
def train_scg(w, X, T, net, sigma, lambda_1, max_epochs, callback=None):
    """Scaled conjugate gradient (Moller, 1993)."""
    w = w.clone()
    N = len(w)
    lam, lam_bar = lambda_1, 0.0
    r = -compute_gradient(w, X, T, net)
    p = r.clone()
    success = True
    k = 1
    with torch.no_grad():
        loss = compute_loss(w, X, T, net).item()
    count = 1
    if callback:
        callback(w)
    delta = 0.0
    while count < max_epochs:
        p_squared = (p @ p).item()
        if p_squared <= 0:
            break
        if success:
            sigma_k = sigma / np.sqrt(p_squared)
            gradient_shifted = compute_gradient(w + sigma_k * p, X, T, net)
            count += 1
            if callback:
                callback(w)
            s = (gradient_shifted + r) / sigma_k
            delta = (p @ s).item()
        delta = delta + (lam - lam_bar) * p_squared
        if delta <= 0:
            lam_bar = 2 * (lam - delta / p_squared)
            delta = -delta + lam * p_squared
            lam = lam_bar
        mu = (p @ r).item()
        if mu == 0 or delta == 0:
            break
        alpha = mu / delta
        with torch.no_grad():
            loss_new = compute_loss(w + alpha * p, X, T, net).item()
        Delta = 2 * delta * (loss - loss_new) / mu ** 2
        if Delta >= 0:
            w = w + alpha * p
            loss = loss_new
            r_new = -compute_gradient(w, X, T, net)
            count += 1
            if callback:
                callback(w)
            lam_bar = 0.0
            success = True
            if k % N == 0:
                p = r_new
            else:
                beta = ((r_new @ r_new - r_new @ r) / mu).item()
                p = r_new + beta * p
            r = r_new
            if Delta >= 0.75:
                lam = lam / 4
        else:
            lam_bar = lam
            success = False
        if Delta < 0.25:
            lam = lam + delta * (1 - Delta) / p_squared
        if torch.linalg.norm(r) < 1e-10:
            break
        k = k + 1
    return w

In [ ]:
def train_leapfrog(w, X, T, net, dt, max_step, m, delta_1, max_epochs, callback=None):
    """LeapFrog LFOP1(b) (Snyman, 1983)."""
    i, j, s, p = 0, 2, 0, 1.0
    w = w.clone()
    a = -compute_gradient(w, X, T, net)
    v = 0.5 * a * dt
    count = 1
    if callback:
        callback(w)
    w_prev, v_prev, w_final = w, v, w
    stop = False
    while count < max_epochs and not stop:
        if torch.linalg.norm(v) * dt < max_step:
            p = p + delta_1
            dt = p * dt
        else:
            v = max_step * v / (dt * torch.linalg.norm(v))
        if s >= m:
            dt = dt / 2
            w = (w + w_prev) / 2
            v = (v + v_prev) / 4
            s = 0
        w_next = w + v * dt
        while count < max_epochs:
            a_next = -compute_gradient(w_next, X, T, net)
            v_next = v + a_next * dt
            count += 1
            if callback:
                callback(w_next)
            w_final = w_next
            if a_next @ a > 0:
                s = 0
            else:
                s = s + 1
                p = 1.0
            if torch.linalg.norm(a_next) <= 1e-5:
                stop = True
                break
            if torch.linalg.norm(v_next) > torch.linalg.norm(v):
                i = 0
                w_prev, v_prev = w, v
                w, v, a = w_next, v_next, a_next
                break
            w_midpoint = (w_next + w) / 2
            i = i + 1
            if i <= j:
                v_next = (v_next + v) / 4
            else:
                v_next = torch.zeros(len(w))
                j = 1
            w_prev, v_prev = w, v
            w, v, a = w_next, v_next, a_next
            w_next = w_midpoint
    return w_final

In [ ]:
DEFAULT_PARAMS = {
    "SGD": {"learning_rate": 0.01, "momentum": 0.9, "batch_size": 32},
    "SCG": {"sigma": 1e-4, "lambda_1": 1e-6},
    "LeapFrog": {"dt": 0.5, "max_step": 1.0, "m": 3, "delta_1": 0.001},
}


def train_network(algorithm, w, X, T, net, params, max_epochs, seed, callback=None):
    """Train with SGD, SCG or LeapFrog and return the final weights."""
    if algorithm == "SGD":
        return train_sgd(w, X, T, net, params["learning_rate"], params["momentum"],
                         params["batch_size"], max_epochs, seed, callback)
    if algorithm == "SCG":
        return train_scg(w, X, T, net, params["sigma"], params["lambda_1"], max_epochs, callback)
    return train_leapfrog(w, X, T, net, params["dt"], params["max_step"], params["m"],
                          params["delta_1"], max_epochs, callback)

## 4. The problems and the folds

In [ ]:
def subsample(X, y, task, max_patterns, seed):
    """Random sample of max_patterns rows. Classification keeps the class proportions."""
    if X.shape[0] <= max_patterns:
        return X, y
    rng = np.random.RandomState(seed)
    if task == "classification":
        keep = []
        for label in np.unique(y):
            index = np.where(y == label)[0]
            take = int(round(max_patterns * len(index) / len(y)))
            keep.append(rng.choice(index, size=take, replace=False))
        keep = np.concatenate(keep)
    else:
        keep = rng.choice(len(y), size=max_patterns, replace=False)
    keep.sort()
    return X[keep], y[keep]


def load_problems(max_patterns=MAX_PATTERNS):
    """Load the six datasets. max_patterns=None keeps every row."""
    raw = [("wine", "classification", *ds.load_wine(return_X_y=True)),
           ("breast_cancer", "classification", *ds.load_breast_cancer(return_X_y=True)),
           ("digits", "classification", *ds.load_digits(return_X_y=True)),
           ("diabetes", "regression", *ds.load_diabetes(return_X_y=True)),
           ("airfoil", "regression", *ds.fetch_openml(data_id=44957, return_X_y=True, as_frame=False)),
           ("california", "regression", *ds.fetch_california_housing(return_X_y=True))]
    problems = []
    for name, task, X, y in raw:
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        if task == "regression":
            y = y.astype(np.float64)
        if max_patterns is not None:
            X, y = subsample(X, y, task, max_patterns, SEED)
        problems.append({"name": name, "task": task, "X": X, "y": y})
    return problems


def make_splits(X, y, task, n_folds, seed):
    """Stratified folds for classification and ordinary folds for regression."""
    if task == "classification":
        return list(StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed).split(X, y))
    return list(KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(X))


def prepare_fold(X, y, train_index, val_index, task):
    """Standardise with the training fold and return tensors."""
    X_train, X_val = X[train_index], X[val_index]
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    if task == "classification":
        classes = list(np.unique(y))
        eye = np.eye(len(classes))
        T_train = eye[[classes.index(label) for label in y[train_index]]]
        T_val = eye[[classes.index(label) for label in y[val_index]]]
    else:
        y_mean, y_std = y[train_index].mean(), y[train_index].std()
        if y_std == 0:
            y_std = 1.0
        T_train = ((y[train_index] - y_mean) / y_std).reshape(-1, 1)
        T_val = ((y[val_index] - y_mean) / y_std).reshape(-1, 1)
    return [torch.tensor(a) for a in (X_train, T_train, X_val, T_val)]


def n_outputs(problem):
    """One output per class for classification and one output for regression."""
    if problem["task"] == "classification":
        return len(np.unique(problem["y"]))
    return 1


problems = load_problems()

## 5. Step 1: the number of hidden units

In [ ]:
def run_sweep(problem, algorithm):
    """Train every candidate size on five folds with the default control parameters."""
    X, y, task = problem["X"], problem["y"], problem["task"]
    rows = []
    splits = make_splits(X, y, task, N_FOLDS, SEED)
    for n_hidden in HIDDEN_UNIT_CANDIDATES:
        for fold, (train_index, val_index) in enumerate(splits):
            X_train, T_train, X_val, T_val = prepare_fold(X, y, train_index, val_index, task)
            net = make_network(X.shape[1], n_hidden, n_outputs(problem), task)
            w = train_network(algorithm, init_weights(net, SEED + fold), X_train, T_train,
                              net, DEFAULT_PARAMS[algorithm], MAX_EPOCHS, SEED + fold)
            rows.append({"problem": problem["name"], "algorithm": algorithm,
                         "hidden_units": n_hidden, "fold": fold,
                         "val_error": compute_error(w, X_val, T_val, net),
                         "train_error": compute_error(w, X_train, T_train, net)})
    return rows


if RUN_SEARCH:
    rows = []
    for problem in problems:
        for algorithm in ALGORITHMS:
            print("sweep:", problem["name"], algorithm)
            rows += run_sweep(problem, algorithm)
    sweep = pd.DataFrame(rows)
    sweep.to_csv(os.path.join(RESULTS, "arch_sweep.csv"), index=False)
else:
    sweep = pd.read_csv(os.path.join(MAIN_RESULTS, "arch_sweep.csv"))

In [ ]:
CHOSEN_HIDDEN = {}
for name in PROBLEMS:
    part = sweep[sweep["problem"] == name]
    means = part.groupby("hidden_units")["val_error"].mean()
    ses = part.groupby("hidden_units")["val_error"].std() / np.sqrt(15)
    lowest = means.idxmin()
    rule = min(h for h in means.index if means[h] <= means[lowest] + ses[lowest])
    CHOSEN_HIDDEN[name] = 2 if name == "diabetes" else rule      # two units for diabetes
print(CHOSEN_HIDDEN)

## 6. Step 2: the control parameters

In [ ]:
PARAM_GRIDS = {
    "SGD": [{"learning_rate": lr, "momentum": mom, "batch_size": batch}
            for lr in [0.001, 0.005, 0.02, 0.05, 0.1, 0.3]
            for mom in [0.0, 0.5, 0.9, 0.99]
            for batch in [16, 64]],
    "SCG": [{"sigma": sigma, "lambda_1": lam}
            for sigma in [1e-5, 1e-4, 1e-3]
            for lam in [1e-8, 1e-6, 1e-4]],
    "LeapFrog": [{"dt": dt, "max_step": max_step, "m": m, "delta_1": 0.001}
                 for dt in [0.1, 0.25, 0.5, 1.0]
                 for max_step in [0.5, 1.0, 2.0]
                 for m in [2, 3, 5]],
}
for algorithm in ALGORITHMS:
    if DEFAULT_PARAMS[algorithm] not in PARAM_GRIDS[algorithm]:
        PARAM_GRIDS[algorithm].insert(0, DEFAULT_PARAMS[algorithm])


def params_to_text(params):
    """Write a setting as text, for example 'sigma=0.0001, lambda_1=1e-06'."""
    return ", ".join(f"{name}={value}" for name, value in params.items())


def tune_one(problem, algorithm, params):
    """Train one setting on the five tuning folds."""
    X, y, task = problem["X"], problem["y"], problem["task"]
    rows = []
    for fold, (train_index, val_index) in enumerate(make_splits(X, y, task, N_FOLDS, TUNE_SEED)):
        X_train, T_train, X_val, T_val = prepare_fold(X, y, train_index, val_index, task)
        net = make_network(X.shape[1], CHOSEN_HIDDEN[problem["name"]], n_outputs(problem), task)
        start = time.perf_counter()
        w = train_network(algorithm, init_weights(net, TUNE_SEED + fold), X_train, T_train,
                          net, params, MAX_EPOCHS, TUNE_SEED + fold)
        error = compute_error(w, X_val, T_val, net)
        rows.append({"problem": problem["name"], "algorithm": algorithm,
                     "params": params_to_text(params), "fold": fold,
                     "val_error": error if np.isfinite(error) else np.inf,
                     "seconds": time.perf_counter() - start})
    return rows


if RUN_SEARCH:
    rows = []
    for problem in problems:
        for algorithm in ALGORITHMS:
            print("tuning:", problem["name"], algorithm)
            for params in PARAM_GRIDS[algorithm]:
                rows += tune_one(problem, algorithm, params)
    tuning = pd.DataFrame(rows)
    tuning.to_csv(os.path.join(RESULTS, "tuning.csv"), index=False)
else:
    tuning = pd.read_csv(os.path.join(MAIN_RESULTS, "tuning.csv"))

means = tuning.groupby(["problem", "algorithm", "params"])["val_error"].mean()
BEST_PARAMS = {}
for name in PROBLEMS:
    for algorithm in ALGORITHMS:
        settings = {params_to_text(p): p for p in PARAM_GRIDS[algorithm]}
        BEST_PARAMS[(name, algorithm)] = settings[means[name][algorithm].idxmin()]
        print(name, algorithm, BEST_PARAMS[(name, algorithm)])

## 7. Step 3: five-fold evaluation on the complete datasets

In [ ]:
final_problems = load_problems(max_patterns=None)
rows = []
for problem in final_problems:
    name, X, y, task = problem["name"], problem["X"], problem["y"], problem["task"]
    splits = make_splits(X, y, task, 5, FINAL_SPLIT_SEED)
    for fold, (train_index, test_index) in enumerate(splits):
        X_train, T_train, X_test, T_test = prepare_fold(X, y, train_index, test_index, task)
        net = make_network(X.shape[1], CHOSEN_HIDDEN[name], n_outputs(problem), task)
        for algorithm in ALGORITHMS:
            start = time.perf_counter()
            w = train_network(algorithm, init_weights(net, FINAL_SPLIT_SEED + fold), X_train, T_train,
                              net, BEST_PARAMS[(name, algorithm)], MAX_EPOCHS, FINAL_SPLIT_SEED + fold)
            seconds = time.perf_counter() - start
            rows.append({"problem": name, "algorithm": algorithm, "fold": fold + 1,
                         "test_error": compute_error(w, X_test, T_test, net),
                         "test_loss": test_loss(w, X_test, T_test, net),
                         "train_error": compute_error(w, X_train, T_train, net),
                         "seconds": seconds})
    print("final:", name)
final = pd.DataFrame(rows)
final.to_csv(os.path.join(RESULTS, "final.csv"), index=False)

summary = final.groupby(["problem", "algorithm"])[["test_error", "test_loss", "train_error", "seconds"]].agg(["mean", "std"])
print(summary.reindex(PROBLEMS, level=0).round(3).to_string())

## 8. Friedman and Nemenyi tests

In [ ]:
k, N = len(ALGORITHMS), len(PROBLEMS)
critical_difference = 2.343 * np.sqrt(k * (k + 1) / (6 * N))
for metric in ["test_loss", "test_error", "seconds"]:
    table = final.groupby(["problem", "algorithm"])[metric].mean().unstack()[ALGORITHMS].loc[PROBLEMS]
    ranks = table.rank(axis=1).mean()
    chi2, p = stats.friedmanchisquare(*[table[a] for a in ALGORITHMS])
    print(f"{metric}: ranks {ranks.round(2).to_dict()}, chi2 = {chi2:.2f}, p = {p:.4f}")
    if p < 0.05:
        for i in range(k):
            for j in range(i + 1, k):
                difference = abs(ranks.iloc[i] - ranks.iloc[j])
                if difference > critical_difference:
                    print(f"   {ALGORITHMS[i]} and {ALGORITHMS[j]} differ: {difference:.2f} > {critical_difference:.3f}")

## 9. Convergence
The final runs again, recording the training loss and the held-out loss after every gradient calculation.

In [ ]:
rows = []
for problem in final_problems:
    name, X, y, task = problem["name"], problem["X"], problem["y"], problem["task"]
    splits = make_splits(X, y, task, 5, FINAL_SPLIT_SEED)
    for fold, (train_index, test_index) in enumerate(splits):
        X_train, T_train, X_test, T_test = prepare_fold(X, y, train_index, test_index, task)
        net = make_network(X.shape[1], CHOSEN_HIDDEN[name], n_outputs(problem), task)
        for algorithm in ALGORITHMS:
            history = []

            def record(w):
                history.append((test_loss(w, X_train, T_train, net), test_loss(w, X_test, T_test, net)))

            train_network(algorithm, init_weights(net, FINAL_SPLIT_SEED + fold), X_train, T_train, net,
                          BEST_PARAMS[(name, algorithm)], MAX_EPOCHS, FINAL_SPLIT_SEED + fold, record)
            history = history[:MAX_EPOCHS]
            while len(history) < MAX_EPOCHS:
                history.append(history[-1])
            for step, (train_loss, heldout_loss) in enumerate(history, start=1):
                rows.append({"problem": name, "algorithm": algorithm, "fold": fold + 1, "step": step,
                             "train_loss": train_loss, "heldout_loss": heldout_loss})
convergence = pd.DataFrame(rows)
convergence.to_csv(os.path.join(RESULTS, "convergence.csv"), index=False)

In [ ]:
for column, log_scale in [("train_loss", True), ("heldout_loss", False)]:
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for ax, name in zip(axes.ravel(), PROBLEMS):
        for algorithm in ALGORITHMS:
            part = convergence[(convergence["problem"] == name) & (convergence["algorithm"] == algorithm)]
            curve = part.groupby("step")[column].mean()
            ax.plot(curve.index, curve.values, color=COLOURS[algorithm], label=algorithm)
        if log_scale:
            ax.set_yscale("log")
        ax.set_title(name)
        ax.set_xlabel("gradient calculations")
        ax.set_ylabel(column)
    axes[0, 0].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS, column + ".png"))
    plt.show()

## 10. Comparison with the NumPy version
If `1_main_experiment.ipynb` has saved its final results, the largest differences are printed here.

In [ ]:
numpy_file = os.path.join(MAIN_RESULTS, "final.csv")
if os.path.exists(numpy_file):
    both = final.merge(pd.read_csv(numpy_file), on=["problem", "algorithm", "fold"], suffixes=("_torch", "_numpy"))
    for metric in ["test_error", "test_loss", "train_error"]:
        both[metric] = (both[metric + "_torch"] - both[metric + "_numpy"]).abs()
    print(both.groupby("algorithm")[["test_error", "test_loss", "train_error"]].max())